In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

# 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。

你可以通过官方应用商店下载我的App来使用我。我很乐意帮助你解答问题、处理文档、进行对话交流等等！

有什么我可以帮你的吗？无论是学习、工作还是生活中的问题，我都很愿意为你提供帮助！✨

In [15]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你以海盗的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

哈哈！我是这片数字海洋里的海盗船长，专门打捞知识宝藏！说吧，伙计，你想知道什么？金银财宝还是藏宝图？

# 2.提示词工程
所谓**提示词工程（Prompt Engineering）**，就是通过优化提示词使模型输出的结果更符合业务需要的过程。




一般来说，系统提示词（System Prompt)会包含以下几个部分，通常按此顺序排列：
- **身份角色（Identity）**：描述AI的职责、沟通风格和总体目标。
- **指令说明（Instructions）**：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- **对话示例（Examples）**：提供可能的输入示例，以及模型期望的输出。
- **背景信息（Context）**：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。


在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML** 标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。




## 2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：


In [16]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在Python中，你可以通过以下几种方式定义字符串变量来记录学校名字：

## 1. 基本定义方法
```python
# 使用单引号
school_name = '清华大学'

# 使用双引号
school_name = "北京大学"

# 使用三引号（多行字符串）
school_name = """复旦大学"""
```

## 2. 常见示例
```python
# 定义学校名字变量
school = "清华大学"

# 打印学校名字
print(f"我的学校是：{school}")

# 多个学校名字
school1 = "北京大学"
school2 = "复旦大学"
school3 = "上海交通大学"
```

## 3. 实际应用示例
```python
# 定义学校信息
school_name = "清华大学"
school_type = "公立大学"
location = "北京市"

# 组合使用
print(f"学校名称：{school_name}")
print(f"学校类型：{school_type}")
print(f"所在城市：{location}")

# 或者创建字典存储
school_info = {
    "name": "清华大学",
    "type": "公立大学",
    "location": "北京市"
}

print(f"学校信息：{school_info['name']}")
```

## 4. 注意事项
- Python中的字符串可以用单引号、双引号或三引号定义
- 变量名建议使用有意义的英文单词，如 `school_name`、`university` 等
- 变量名区分大小写，`school` 和 `School` 是不同的变量

最简单的定义方式就是：
```python
school = "你的学校名字"
```

你需要记录哪个具体的学校名字呢？我可以帮你写出更具体的代码。

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [17]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


school_name = "黑马程序员"


## 2.2.对话示例（Few-Shot examples）

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。


In [18]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在未来的金星殖民纪元，人类在金星大气层中建造了悬浮都市群。其中最为宏伟的，是被称为 **“云顶天都”（Nephos Prime）** 的空中首都。

### 🌌 云顶天都 · 悬浮奇迹
这座首都并非建立在金星炽热的地表，而是漂浮在距地表约50公里的大气层中——这里的温度、气压与地球相近。整座城市由数千个**反重力平台**和**生态穹顶**连接而成，外观如同巨大的水晶莲花，在金色的硫酸云层之上反射着恒星的光芒。

### 🏛️ 城市结构
- **晨曦环带**：政治中心，金星联邦议会大厦形如旋转的钻石，悬浮于主平台之上
- **硅谷云台**：科技中枢，数据流在城市底部形成发光的“知识瀑布”
- **生态穹顶群**：每个穹顶模拟地球不同生态区，通过气候薄膜维持生存环境
- **星港矩阵**：飞船停泊在云层中的磁悬浮轨道，进出时会在云中划出彩虹般的光痕

### 🌠 独特现象
由于金星大气富含二氧化碳和硫酸微粒，每当日落时分（虽然金星自转极慢），城市周围会出现**双虹蚀光**——大气折射形成两道方向相反的彩虹光环，将整座城市笼罩在幻彩之中。

这座悬浮首都象征着人类在极端环境中创造宜居世界的智慧，成为太阳系中最具未来主义色彩的政治与文化中心。每当星际飞船穿越金色云层靠近时，乘客们总会惊叹：“看，云中之国在发光。”

（注：此为科幻设定，实际金星表面温度约460°C，大气压力是地球的92倍，目前尚未有生命存在或人类殖民）

In [19]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


**云冕城（Aphrodia）**——悬浮于金星硫酸云层上空的浮岛集群都市，由反重力引擎与太阳帆阵列共同维持，其核心是一座能将大气二氧化碳转化为钻石穹顶的巨型催化塔。

## 2.3.结构化输出
模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出


In [20]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么?', additional_kwargs={}, response_metadata={}, id='e37bbf5f-0cc3-4106-8229-f66cf6560935'), AIMessage(content='{\n    "name": "硫磺城（Sulfura）",\n    "location": "悬浮于金星浓厚大气层中距地表约50公里的高空，由巨大的反重力浮空平台群构成",\n    "vibe": "高压、炽热、坚韧",\n    "economy": "大气资源提炼（二氧化碳、硫酸）、极端环境材料制造、太阳能巨型阵列"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 141, 'total_tokens': 222, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '6762862e-2f37-40ab-bbef-23a07a92b134', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d230c-af6c-7992-b9dd-04fc6b9d4f02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_token

In [21]:
print(response['messages'][-1].content)

{
    "name": "硫磺城（Sulfura）",
    "location": "悬浮于金星浓厚大气层中距地表约50公里的高空，由巨大的反重力浮空平台群构成",
    "vibe": "高压、炽热、坚韧",
    "economy": "大气资源提炼（二氧化碳、硫酸）、极端环境材料制造、太阳能巨型阵列"
}


### b.基于Model的结构化输出

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式


In [22]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [23]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
# 输出结果
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='bdc4c411-a8d5-4351-939a-d6f6e5689a54'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 355, 'total_tokens': 489, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '668ea94a-cfd7-4547-8039-f82b22edf755', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d230e-e690-7652-ad8b-87703c39fbb5-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '月宫', 'location': '月球南极-艾特肯盆地边缘', 'vibe': '未来主义与古典东方美学融合，低重力环境下的优雅建筑，透明穹顶下的花园城市', 'economy': '氦-3开采与精炼、月球旅游、零重力制造、科学研究、稀有金属贸易'}, 'id': 'call_00_i68mFAfCzaCGVA8POKixhIxO', 'type': 'tool_call'}], i

In [24]:
city = response['structured_response']
city

CapitalInfo(name='月宫', location='月球南极-艾特肯盆地边缘', vibe='未来主义与古典东方美学融合，低重力环境下的优雅建筑，透明穹顶下的花园城市', economy='氦-3开采与精炼、月球旅游、零重力制造、科学研究、稀有金属贸易')

In [25]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

月宫位于月球南极-艾特肯盆地边缘，是一座未来主义与古典东方美学融合，低重力环境下的优雅建筑，透明穹顶下的花园城市的城市，其主要产业包括氦-3开采与精炼、月球旅游、零重力制造、科学研究、稀有金属贸易。


## 2.4.完整示例

接下来，看一个包含角色、指令、示例的完整提示词示例：


In [ ]:
# 舆情分析案例
# 根据用户对商品的评价判断是好评、差评、中评中的哪一个

system_prompt = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你们家产品质量真是好啊，我用了两天就坏了！！")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)
